# BeyondKVTransfer Q1-Q6 Reference Analysis

Set `TRACE_PATH` to a raw trace directory containing `manifest.json`, or set `INDEX_PATH` to a directory produced by `analysis/build_index.py`. The notebook renders one canonical figure for each research question in `DESIGN.md`.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import pandas as pd

from analysis.lib.critical_path import critical_path_attribution, request_lifecycle, scheduler_tail_latency
from analysis.lib.load import load_index, load_trace
from analysis.lib.prefetch import prefetch_slack
from analysis.lib.reuse import reuse_distance, tier_residency

TRACE_PATH = None  # e.g. '../bkvt_traces/<trace_id>'
INDEX_PATH = None  # e.g. '../analysis/index'

if INDEX_PATH:
    tables = load_index(INDEX_PATH)
elif TRACE_PATH:
    tables = load_trace(TRACE_PATH)
else:
    raise ValueError('Set TRACE_PATH or INDEX_PATH before running the notebook.')

requests = tables.get('request', pd.DataFrame())
tokens = tables.get('token', pd.DataFrame())
kv_blocks = tables.get('kv_block', pd.DataFrame())
transfers = tables.get('transfer', pd.DataFrame())
metadata = tables.get('metadata', pd.DataFrame())
sys_counters = tables.get('sys_counter', pd.DataFrame())

lifecycle = tables.get('request_lifecycle') if 'request_lifecycle' in tables and not tables['request_lifecycle'].empty else request_lifecycle(requests, tokens)
critical = tables.get('critical_path') if 'critical_path' in tables and not tables['critical_path'].empty else critical_path_attribution(lifecycle, transfers, metadata)
slack = tables.get('prefetch_slack') if 'prefetch_slack' in tables and not tables['prefetch_slack'].empty else prefetch_slack(transfers)
reuse = tables.get('reuse_distance') if 'reuse_distance' in tables and not tables['reuse_distance'].empty else reuse_distance(kv_blocks)
residency = tables.get('tier_residency') if 'tier_residency' in tables and not tables['tier_residency'].empty else tier_residency(kv_blocks)
sched = tables.get('scheduler_tail_latency') if 'scheduler_tail_latency' in tables and not tables['scheduler_tail_latency'].empty else scheduler_tail_latency(metadata, lifecycle)

def ns_to_s(series):
    return series / 1_000_000_000

def empty_figure(title):
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.set_title(title)
    ax.text(0.5, 0.5, 'No matching records in this trace', ha='center', va='center')
    ax.set_axis_off()
    return fig

plt.rcParams['figure.dpi'] = 120

## Q1. Remote KV Data Path

In [ ]:
if transfers.empty or 'bytes' not in transfers:
    empty_figure('Q1: Remote KV bytes over time')
else:
    q1 = transfers[transfers['subtype'].isin(['start', 'end'])].copy()
    q1 = q1.dropna(subset=['bytes'])
    if q1.empty:
        empty_figure('Q1: Remote KV bytes over time')
    else:
        q1['t_s'] = ns_to_s(q1['ts_ns'] - q1['ts_ns'].min())
        q1['transport'] = q1['transport'].fillna('unknown') if 'transport' in q1 else 'unknown'
        q1['bucket_s'] = q1['t_s'].round(1)
        pivot = q1.pivot_table(index='bucket_s', columns='transport', values='bytes', aggfunc='sum').fillna(0)
        ax = pivot.plot.area(figsize=(9, 4))
        ax.set_title('Q1: Remote KV bytes over time')
        ax.set_xlabel('Trace time (s)')
        ax.set_ylabel('Bytes')

## Q2. Remote Metadata Path

In [ ]:
if metadata.empty or 'duration_ns' not in metadata:
    empty_figure('Q2: Metadata duration CDF')
else:
    q2 = metadata.dropna(subset=['duration_ns']).copy()
    if q2.empty:
        empty_figure('Q2: Metadata duration CDF')
    else:
        fig, ax = plt.subplots(figsize=(9, 4))
        for subtype, group in q2.groupby('subtype'):
            values = group['duration_ns'].sort_values().reset_index(drop=True)
            y = (values.index + 1) / len(values)
            ax.plot(values / 1_000, y, label=subtype)
        ax.set_title('Q2: Metadata operation duration CDF')
        ax.set_xlabel('Duration (us)')
        ax.set_ylabel('CDF')
        ax.legend(loc='lower right', fontsize=8)

## Q3. Critical Path

In [ ]:
q3 = critical.dropna(subset=['duration_ns']) if not critical.empty else critical
if q3.empty:
    empty_figure('Q3: Critical-path attribution')
else:
    top = q3.groupby('stage')['duration_ns'].sum().sort_values(ascending=False)
    ax = (top / 1_000_000).plot.bar(figsize=(8, 4))
    ax.set_title('Q3: Critical-path attribution across workload')
    ax.set_xlabel('Stage')
    ax.set_ylabel('Total duration (ms)')

## Q4. Reuse / Locality

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
if reuse.empty:
    axes[0].text(0.5, 0.5, 'No prefix-hit reuse records', ha='center', va='center')
    axes[0].set_axis_off()
else:
    values = reuse['reuse_distance_ns'].sort_values().reset_index(drop=True)
    axes[0].plot(values / 1_000_000, (values.index + 1) / len(values))
    axes[0].set_xlabel('Reuse distance (ms)')
    axes[0].set_ylabel('CDF')
if residency.empty:
    axes[1].text(0.5, 0.5, 'No tier residency intervals', ha='center', va='center')
    axes[1].set_axis_off()
else:
    (residency.groupby('tier_after')['residency_ns'].sum() / 1_000_000).sort_values().plot.barh(ax=axes[1])
    axes[1].set_xlabel('Residency (ms)')
axes[0].set_title('Reuse-distance CDF')
axes[1].set_title('Per-tier residency')
fig.suptitle('Q4: Reuse / locality')
fig.tight_layout()

## Q5. Prefetchability

In [ ]:
if slack.empty:
    empty_figure('Q5: Prefetch slack')
else:
    ax = (slack['prefetch_slack_ns'] / 1_000_000).plot.hist(bins=50, figsize=(8, 4))
    ax.set_title('Q5: Prefetch slack')
    ax.set_xlabel('started_ts_ns - earliest_known_ts_ns (ms)')
    ax.set_ylabel('Transfer count')

## Q6. Scheduling Impact

In [ ]:
if sched.empty or 'queue_depth' not in sched:
    empty_figure('Q6: Scheduler queue depth vs tail latency')
else:
    q6 = sched.dropna(subset=['queue_depth', 'workload_p99_ns']).copy()
    if q6.empty:
        empty_figure('Q6: Scheduler queue depth vs tail latency')
    else:
        ax = q6.plot.scatter(x='queue_depth', y='workload_p99_ns', figsize=(8, 4))
        ax.set_title('Q6: Scheduler queue depth vs tail latency')
        ax.set_xlabel('Queue depth at scheduler decision')
        ax.set_ylabel('Workload P99 end-to-end latency (ns)')